In [5]:
# Before starting the fine-tuning process, make sure you have the necessary environment set up. 
# You will need access to a pretrained LLM (such as GPT, BERT, or T5) and a task-specific dataset.

# Install necessary libraries
#!pip install transformers datasets

# Import relevant modules
# For Step 1
from transformers import AutoModelForSequenceClassification, AutoTokenizer
# For Step 2
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import pandas as pd
from datasets import Dataset
# For Step 3
import os
import torch
from transformers import Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score

# Step 1: Set up the environment
# Load a pretrained model and tokenizer (e.g., BERT for sequence classification)
model_name = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Step 2: Prepare the dataset
# 1. Load your dataset, ensuring that it is properly formatted for the task (e.g., text and labels for classification).
# 2. Split your dataset into training and test sets.
# 3. Preprocess the text using the tokenizer of the pretrained model.


# Load the IMDb dataset
dataset = load_dataset('imdb')

# Convert dataset to Pandas DataFrame
df = dataset['train'].to_pandas()

# Perform train-test split
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)

# Tokenize the dataset
def preprocess_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True)

# Convert back to Hugging Face dataset

train_data = Dataset.from_pandas(train_data)
test_data = Dataset.from_pandas(test_data)

# Apply preprocessing
train_data = train_data.map(preprocess_function, batched=True)
test_data = test_data.map(preprocess_function, batched=True)

# Step 3: Fine-tune the LLM
# Now, you will fine-tune the pretrained LLM on your dataset. This process involves training the model on the task-specific data and optimizing its parameters for your task.
# 1. Set up the training arguments, such as learning rate, number of epochs, and batch size.
# 2. Fine-tune the model on the training data while validating it on the validation set.
# Disable parallelism warning and MLflow logging
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["MLFLOW_TRACKING_URI"] = "disable"
os.environ["HF_MLFLOW_LOGGING"] = "false"

# Ensure CPU usage if no GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load a smaller, faster model like DistilBERT
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
model.to(device)

# Use a subset of the dataset to speed up training
train_data = train_data.select(range(1000))  # Select 1000 samples for training
test_data = test_data.select(range(200))     # Select 200 samples for evaluation

# Define a function to compute metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy}

# Set up training arguments for faster training
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",           # Changed from "steps" to "epoch"
    learning_rate=2e-5,
    per_device_train_batch_size=8,   
    num_train_epochs=1,              
    weight_decay=0,                  
    logging_steps=50,                # Adjusted to be smaller than total steps
    save_steps=1000,                 
    save_total_limit=1,              
    gradient_accumulation_steps=1,   
    fp16=False,                      
    report_to="none",
)                

# Define the Trainer for fine-tuning
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics,
)

# Fine-tune the model
train_output = trainer.train()

# Step 4: Evaluate the fine-tuned model
# Once the model is fine-tuned, it's essential to evaluate its performance on the test set using appropriate metrics, such as accuracy, precision, recall, and F1 score.
# 1. Use the test dataset to evaluate the model's performance.
# 2. Calculate the evaluation metrics and analyze the results.

# Get evaluation metrics from the trainer's log history
# Since eval_strategy="epoch", evaluation was performed at the end of training
eval_results = [log for log in trainer.state.log_history if 'eval_loss' in log]
if eval_results:
    final_eval = eval_results[-1]  # Get the last evaluation
    print(f"Evaluation Loss: {final_eval['eval_loss']:.4f}")
    print(f"Accuracy: {final_eval['eval_accuracy']:.4f}")
else:
    print("No evaluation results found in training history.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/Users/peter.horstedt/git/AI-and-Machine-Learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be use

Epoch,Training Loss,Validation Loss,Accuracy
1,0.463031,0.354885,0.865000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/peter.horstedt/git/AI-and-Machine-Learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Evaluation Loss: 0.3549
Accuracy: 0.8650
